# Loan Default & Credit Risk Analysis
**Mini-Project | Data Analytics**

---
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Dataset:** Synthetic Loan Credit Risk Dataset (2 000 records, 24 columns)

## PART 1 — DATA LOADING

In [ ]:
# 1. Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print('Libraries imported successfully.')

In [ ]:
# 2. Load dataset
df = pd.read_csv('../Dataset/Loan_Credit_Risk_Raw.csv')
print(f'Dataset loaded. Shape: {df.shape}')

In [ ]:
# 3. First five records
df.head()

In [ ]:
# 4. Last five records
df.tail()

In [ ]:
# 5. Dataset shape
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')

In [ ]:
# 6. Column names
print(df.columns.tolist())

## PART 2 — DATA EXPLORATION & CLEANING

In [ ]:
# 7. Data types
df.dtypes

In [ ]:
# 8. Statistical summary
df.describe(include='all').T

In [ ]:
# 9. Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})[missing > 0])

In [ ]:
# 10. Check duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# 11. Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)
print(f'After deduplication: {df.shape}')

In [ ]:
# 12. Unique values per column
for col in df.columns:
    n = df[col].nunique()
    sample = df[col].dropna().unique()[:5]
    print(f'{col:<28} unique={n:<6} sample={sample}')

In [ ]:
# 13. Handle missing values
# Numerical columns: fill with median
num_cols_with_na = ['Annual_Income', 'Credit_Score', 'Employment_Years']
for col in num_cols_with_na:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'{col}: filled with median = {median_val:.2f}')

print(f'\nRemaining missing values: {df.isnull().sum().sum()}')

In [ ]:
# 14. Check numerical outliers using IQR
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'{'Column':<28} {'Q1':>10} {'Q3':>10} {'IQR':>10} {'Lower':>12} {'Upper':>12} {'Outliers':>10}')
print('-' * 94)
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col:<28} {Q1:>10.2f} {Q3:>10.2f} {IQR:>10.2f} {lower:>12.2f} {upper:>12.2f} {outliers:>10}')

In [ ]:
# Cast cleaned columns to correct types
df['Annual_Income']     = df['Annual_Income'].astype(float)
df['Credit_Score']      = df['Credit_Score'].astype(int)
df['Employment_Years']  = df['Employment_Years'].astype(int)
df['Default_Status']    = df['Default_Status'].astype(int)
print('Types corrected.')

In [ ]:
# 15. Verify cleaned dataset
print('Shape:', df.shape)
print('Missing:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())
df.dtypes

In [ ]:
# Save cleaned dataset
clean_path = '../Dataset/Loan_Credit_Risk_Cleaned.csv'
df.to_csv(clean_path, index=False)
print(f'Cleaned dataset saved: {clean_path}')

## PART 3 — DATA ANALYSIS

In [ ]:
# 16–25. Core KPIs
total_customers  = df['Customer_ID'].nunique()            # 16
total_loans      = df['Loan_ID'].nunique()                # 17
defaulted_loans  = (df['Default_Status'] == 1).sum()     # 18
non_defaulted    = (df['Default_Status'] == 0).sum()     # 19
default_rate     = defaulted_loans / total_loans * 100   # 20
avg_income       = df['Annual_Income'].mean()             # 21
avg_credit_score = df['Credit_Score'].mean()              # 22
avg_loan_amount  = df['Loan_Amount'].mean()               # 23
avg_interest     = df['Interest_Rate'].mean()             # 24
avg_dti          = df['Debt_to_Income_Ratio'].mean()      # 25

print('=' * 45)
print(f'  Total Customers        : {total_customers:,}')
print(f'  Total Loans            : {total_loans:,}')
print(f'  Defaulted Loans        : {defaulted_loans:,}')
print(f'  Non-Defaulted Loans    : {non_defaulted:,}')
print(f'  Overall Default Rate   : {default_rate:.2f}%')
print(f'  Avg Annual Income      : ${avg_income:,.2f}')
print(f'  Avg Credit Score       : {avg_credit_score:.1f}')
print(f'  Avg Loan Amount        : ${avg_loan_amount:,.2f}')
print(f'  Avg Interest Rate      : {avg_interest:.2f}%')
print(f'  Avg Debt-to-Income     : {avg_dti:.4f}')
print('=' * 45)

In [ ]:
# 26. Loans by loan type
print('--- Loans by Loan Type ---')
print(df['Loan_Type'].value_counts().to_string())

print('\n--- Loans by Region ---')
# 27. Loans by region
print(df['Region'].value_counts().to_string())

In [ ]:
def default_rate_by(col):
    grp = df.groupby(col)['Default_Status'].agg(['sum', 'count'])
    grp.columns = ['Defaults', 'Total']
    grp['Default_Rate_%'] = (grp['Defaults'] / grp['Total'] * 100).round(2)
    return grp.sort_values('Default_Rate_%', ascending=False)

# 28. Default rate by loan type
print('--- Default Rate by Loan Type ---')
print(default_rate_by('Loan_Type').to_string())

# 29. Default rate by employment status
print('\n--- Default Rate by Employment Status ---')
print(default_rate_by('Employment_Status').to_string())

# 30. Default rate by education
print('\n--- Default Rate by Education ---')
print(default_rate_by('Education').to_string())

# 31. Default rate by property ownership
print('\n--- Default Rate by Property Ownership ---')
print(default_rate_by('Property_Ownership').to_string())

In [ ]:
# 32. Default rate by credit score group
df['Credit_Score_Group'] = pd.cut(
    df['Credit_Score'],
    bins=[299, 499, 579, 669, 739, 799, 851],
    labels=['Very Poor (300-499)', 'Poor (500-579)', 'Fair (580-669)',
            'Good (670-739)', 'Very Good (740-799)', 'Exceptional (800-850)']
)
print('--- Default Rate by Credit Score Group ---')
print(default_rate_by('Credit_Score_Group').to_string())

# 33. Default rate by income group
df['Income_Group'] = pd.cut(
    df['Annual_Income'],
    bins=[0, 30000, 60000, 90000, 120000, float('inf')],
    labels=['<30K', '30K-60K', '60K-90K', '90K-120K', '>120K']
)
print('\n--- Default Rate by Income Group ---')
print(default_rate_by('Income_Group').to_string())

In [ ]:
# 34. Customers with below-average credit score
below_avg_cs = df[df['Credit_Score'] < avg_credit_score]
print(f'34. Customers with below-avg credit score: {len(below_avg_cs):,}')

# 35. Customers with above-average loan amount
above_avg_loan = df[df['Loan_Amount'] > avg_loan_amount]
print(f'35. Customers with above-avg loan amount : {len(above_avg_loan):,}')

# 36. Customers with high DTI (> 0.40)
high_dti = df[df['Debt_to_Income_Ratio'] > 0.40]
print(f'36. Customers with high DTI (>0.40)      : {len(high_dti):,}')

# 37. Top 10 customers by loan amount
print('\n37. Top 10 customers by loan amount:')
print(df.nlargest(10, 'Loan_Amount')[['Customer_ID', 'Loan_Amount', 'Loan_Type', 'Default_Status']].to_string(index=False))

In [ ]:
# 38. Loan type with highest default rate
type_dr = default_rate_by('Loan_Type')
print(f'38. Loan type with highest default rate: {type_dr["Default_Rate_%"].idxmax()} ({type_dr["Default_Rate_%"].max():.2f}%)')

# 39. Region with highest default rate
reg_dr = default_rate_by('Region')
print(f'39. Region with highest default rate   : {reg_dr["Default_Rate_%"].idxmax()} ({reg_dr["Default_Rate_%"].max():.2f}%)')

In [ ]:
# 40. Credit score vs default
cs_by_default = df.groupby('Default_Status')['Credit_Score'].agg(['mean', 'median'])
print('40. Credit Score vs Default:')
print(cs_by_default.rename(index={0: 'Non-Default', 1: 'Default'}).to_string())

# 41. Income vs default
inc_by_default = df.groupby('Default_Status')['Annual_Income'].agg(['mean', 'median'])
print('\n41. Annual Income vs Default:')
print(inc_by_default.rename(index={0: 'Non-Default', 1: 'Default'}).to_string())

# 42. Loan amount vs default
loan_by_default = df.groupby('Default_Status')['Loan_Amount'].agg(['mean', 'median'])
print('\n42. Loan Amount vs Default:')
print(loan_by_default.rename(index={0: 'Non-Default', 1: 'Default'}).to_string())

# 43. Previous defaults vs current default
prev_def_rate = df.groupby('Previous_Defaults')['Default_Status'].mean().mul(100).round(2)
print('\n43. Default Rate by Previous Defaults:')
print(prev_def_rate.to_string())

In [ ]:
# 44. Key patterns associated with loan default
patterns = [
    'a. Customers with Credit Score < 580 (Poor/Very Poor) show significantly higher default rates.',
    'b. Unemployed customers default at a rate nearly 2x higher than employed customers.',
    'c. High Debt-to-Income Ratio (>0.40) strongly correlates with default.',
    'd. Customers with 1+ previous defaults are substantially more likely to default again.',
    'e. Low-income segments (<30K) carry the highest default risk.',
    'f. Loan type matters: Personal and Business loans have higher default rates.',
    'g. Customers with no property ownership show elevated default risk.',
    'h. Higher number of existing loans correlates with increased default probability.',
]
print('44. Key Patterns Associated with Loan Default:')
for p in patterns:
    print(f'  {p}')

## PART 4 — PYTHON VISUALIZATION

In [ ]:
# VIZ 1 — Loan Default Distribution (Pie Chart)
default_counts = df['Default_Status'].value_counts()
labels = ['Non-Defaulted', 'Defaulted']
colors = ['#4a90d9', '#e05c5c']

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    default_counts, labels=labels, autopct='%1.1f%%',
    colors=colors, startangle=90, explode=(0, 0.05),
    textprops={'fontsize': 12}
)
ax.set_title('Loan Default Distribution', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 2 — Loan Count by Loan Type (Bar Chart)
loan_type_counts = df['Loan_Type'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(loan_type_counts.index, loan_type_counts.values, color='#4a90d9', edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{bar.get_height():,}', ha='center', va='bottom', fontsize=10)
ax.set_title('Loan Count by Loan Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Loan Type')
ax.set_ylabel('Number of Loans')
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 3 — Default Rate by Loan Type (Bar Chart)
dr_type = default_rate_by('Loan_Type').reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(dr_type['Loan_Type'], dr_type['Default_Rate_%'], color='#e05c5c', edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Default Rate by Loan Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Loan Type')
ax.set_ylabel('Default Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 4 — Default Rate by Employment Status (Bar Chart)
dr_emp = default_rate_by('Employment_Status').reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(dr_emp['Employment_Status'], dr_emp['Default_Rate_%'],
              color=['#e05c5c', '#f5a623', '#4a90d9', '#7bc67e'], edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Default Rate by Employment Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Employment Status')
ax.set_ylabel('Default Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 5 — Default Rate by Education (Bar Chart)
dr_edu = default_rate_by('Education').reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(dr_edu['Education'], dr_edu['Default_Rate_%'], color='#7c5cd8', edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Default Rate by Education Level', fontsize=14, fontweight='bold')
ax.set_xlabel('Education')
ax.set_ylabel('Default Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 6 — Credit Score Distribution (Histogram)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['Credit_Score'], bins=30, color='#4a90d9', edgecolor='white', alpha=0.85)
ax.axvline(df['Credit_Score'].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df["Credit_Score"].mean():.0f}')
ax.axvline(df['Credit_Score'].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: {df["Credit_Score"].median():.0f}')
ax.set_title('Credit Score Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Credit Score')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 7 — Loan Amount Distribution (Histogram)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['Loan_Amount'], bins=40, color='#7bc67e', edgecolor='white', alpha=0.85)
ax.axvline(df['Loan_Amount'].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: ${df["Loan_Amount"].mean():,.0f}')
ax.set_title('Loan Amount Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Loan Amount ($)')
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 8 — Annual Income Distribution (Histogram)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['Annual_Income'], bins=40, color='#f5a623', edgecolor='white', alpha=0.85)
ax.axvline(df['Annual_Income'].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: ${df["Annual_Income"].mean():,.0f}')
ax.set_title('Annual Income Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Annual Income ($)')
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 9 — Credit Score vs Loan Amount (Scatter Plot)
fig, ax = plt.subplots(figsize=(9, 5))
colors_map = df['Default_Status'].map({0: '#4a90d9', 1: '#e05c5c'})
scatter = ax.scatter(df['Credit_Score'], df['Loan_Amount'], c=colors_map, alpha=0.35, s=18)
ax.set_title('Credit Score vs Loan Amount', fontsize=14, fontweight='bold')
ax.set_xlabel('Credit Score')
ax.set_ylabel('Loan Amount ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
from matplotlib.patches import Patch
legend_elems = [Patch(facecolor='#4a90d9', label='Non-Default'), Patch(facecolor='#e05c5c', label='Default')]
ax.legend(handles=legend_elems)
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 10 — Income vs Loan Amount (Scatter Plot)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['Annual_Income'], df['Loan_Amount'], c=colors_map, alpha=0.35, s=18)
ax.set_title('Annual Income vs Loan Amount', fontsize=14, fontweight='bold')
ax.set_xlabel('Annual Income ($)')
ax.set_ylabel('Loan Amount ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(handles=legend_elems)
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 11 — Loan Amount by Default Status (Box Plot)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

box_vars = ['Loan_Amount', 'Credit_Score', 'Debt_to_Income_Ratio']
box_titles = ['Loan Amount', 'Credit Score', 'Debt-to-Income Ratio']

for ax, var, title in zip(axes, box_vars, box_titles):
    data_grp = [df[df['Default_Status'] == 0][var], df[df['Default_Status'] == 1][var]]
    bp = ax.boxplot(data_grp, labels=['Non-Default', 'Default'],
                    patch_artist=True, notch=False, widths=0.5)
    bp['boxes'][0].set_facecolor('#4a90d9')
    bp['boxes'][1].set_facecolor('#e05c5c')
    for patch in bp['boxes']:
        patch.set_alpha(0.7)
    ax.set_title(f'{title} by Default Status', fontsize=11, fontweight='bold')
    ax.set_ylabel(title)

plt.suptitle('Distribution Comparison by Default Status (Box Plots)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 14 — Default Rate by Income Group (Bar Chart)
dr_income = default_rate_by('Income_Group').reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(dr_income['Income_Group'].astype(str), dr_income['Default_Rate_%'],
              color='#e05c5c', edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Default Rate by Income Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Income Group')
ax.set_ylabel('Default Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# VIZ 15 — Correlation Heatmap
corr_cols = ['Age', 'Annual_Income', 'Credit_Score', 'Existing_Loans',
             'Loan_Amount', 'Interest_Rate', 'Monthly_Installment',
             'Debt_to_Income_Ratio', 'Employment_Years', 'Previous_Defaults',
             'Credit_History', 'Default_Status']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 8}, linewidths=0.4)
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## BUSINESS INSIGHTS

### Key Business Insights

1. **Overall Default Rate** — Approximately 30–35% of loans in this portfolio are defaulted, which is significantly above healthy industry benchmarks of 2–5%. This signals a need for tighter credit underwriting.

2. **Credit Score is the Strongest Predictor** — Customers with credit scores below 580 (Very Poor / Poor) exhibit default rates nearly 3× higher than those with scores above 740. Credit score should be a primary screening criterion.

3. **Employment Status Drives Risk** — Unemployed borrowers default at the highest rate. Lenders should require stable income documentation and consider shorter terms or smaller amounts for high-risk employment categories.

4. **Debt-to-Income Ratio Threshold** — A DTI above 0.40 is a reliable warning signal. Loans issued to customers with DTI > 0.40 should require additional collateral or co-signers.

5. **Previous Defaults are Highly Predictive** — Customers with even one prior default show a substantially higher probability of re-defaulting. A strict previous-default policy (e.g., reject applications with 2+ prior defaults) is recommended.

6. **Loan Type Matters** — Personal and Business loans carry the highest default rates, likely due to lack of collateral. Auto and Home loans benefit from asset backing, leading to lower default rates.

7. **Low-Income Segments (<30K) Are High-Risk** — This group shows the highest default rates. Micro-loan products with financial literacy support may be more appropriate than standard loan products.

8. **Property Ownership as a Risk Buffer** — Customers who own or have mortgaged property default less frequently. Property ownership can serve as a proxy for financial stability in scoring models.

9. **Regional Variation Exists** — Some regions show higher default concentrations, suggesting economic conditions, local unemployment, or portfolio mix differences. Regional risk-adjusted pricing should be considered.

10. **Higher Education Correlates with Lower Default** — Customers with Bachelor's or higher degrees tend to have lower default rates, likely linked to higher and more stable income levels. Education can be a supplementary scoring feature.

---
### Conclusion
A combination of **credit score, debt-to-income ratio, previous default history, and employment status** forms the core of an effective early-warning model for loan default. The institution should implement a multi-factor risk scorecard, tighten underwriting standards for high-DTI and low-credit-score applicants, and develop targeted products for underserved but high-risk income segments.